# Project 82 — Local Invoice Extraction Copilot
## OCR Text → Structured Invoice → Validation → Export

**Stack:** LangChain · Ollama · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Simulate OCR Output

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd
from pathlib import Path

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

# Simulated OCR text from scanned invoices
ocr_outputs = [
    {
        "file": "invoice_001.pdf",
        "text": """INVOICE
CloudTech Solutions LLC
123 Innovation Drive, Austin TX 78701

Bill To: DataDriven Inc.
Invoice #: INV-2025-0042
Date: February 10, 2025
Due: March 10, 2025

Description                    Qty    Rate       Amount
API Gateway Setup               1    $2,500     $2,500.00
Cloud Migration (hrs)          40      $150     $6,000.00
Security Audit                  1    $3,000     $3,000.00

                              Subtotal:         $11,500.00
                              Tax (8.25%):         $948.75
                              TOTAL:            $12,448.75

Payment Terms: Net 30
Bank: First National, Acct: ****4521"""
    },
    {
        "file": "invoice_002.pdf",
        "text": """DataPipe Corp --- Invoice
To: Acme Analytics
INV# DP-8891  |  Date: 2025-01-15

- ETL Pipeline Development  120hrs x $125  = $15,000
- Data Quality Module        1 unit         = $4,500
- Training (2 days)          2 x $2,000     = $4,000

Subtotal: $23,500.00
Discount (10%): -$2,350.00
Total Due: $21,150.00
Due by: Feb 15, 2025"""
    },
    {
        "file": "receipt_003.jpg",
        "text": """Quick Mart
Date: 03/05/2025 14:32
Cashier: Mike

Coffee Large         $4.99
Sandwich             $8.49
Water                $1.99
Cookie               $2.49

Subtotal            $17.96
Tax                  $1.48
Total               $19.44
VISA ****7812"""
    },
]
print(f"OCR documents to process: {len(ocr_outputs)}")

## Step 2 — Define Extraction Schemas

In [ ]:
class LineItem(BaseModel):
    description: str
    quantity: float = 1
    unit_price: float = 0
    amount: float

class InvoiceData(BaseModel):
    document_type: str = Field(description="invoice, receipt, purchase_order")
    invoice_number: str = ""
    date: str
    due_date: str = ""
    vendor_name: str
    vendor_address: str = ""
    bill_to: str = ""
    line_items: list[LineItem]
    subtotal: float
    tax: float = 0
    discount: float = 0
    total: float
    payment_method: str = ""
    currency: str = "USD"

extractor = llm.with_structured_output(InvoiceData)
print("Schema: InvoiceData with LineItem sub-model")
print(f"Fields: {len(InvoiceData.model_fields)}")

## Step 3 — Extract All Invoices

In [ ]:
extracted = []
for doc in ocr_outputs:
    try:
        invoice = extractor.invoke(
            f"Extract structured invoice data from this OCR text:\n\n{doc['text']}"
        )
        extracted.append({"file": doc["file"], "data": invoice, "error": None})
        print(f"✓ {doc['file']}: {invoice.vendor_name} | "
              f"{len(invoice.line_items)} items | ${invoice.total:,.2f}")
    except Exception as e:
        extracted.append({"file": doc["file"], "data": None, "error": str(e)})
        print(f"✗ {doc['file']}: {e}")

## Step 4 — Validation & Anomaly Detection

In [ ]:
print("VALIDATION REPORT")
print("=" * 60)
for inv in extracted:
    if inv["data"] is None:
        print(f"✗ {inv['file']}: extraction failed — {inv['error']}")
        continue
    d = inv["data"]
    issues = []

    # Check math
    calc_subtotal = sum(i.amount for i in d.line_items)
    if abs(calc_subtotal - d.subtotal) > 1.0:
        issues.append(f"Subtotal mismatch: items sum to ${calc_subtotal:.2f}, stated ${d.subtotal:.2f}")

    # Check total = subtotal + tax - discount
    expected_total = d.subtotal + d.tax - d.discount
    if abs(expected_total - d.total) > 1.0:
        issues.append(f"Total mismatch: expected ${expected_total:.2f}, stated ${d.total:.2f}")

    # Check missing fields
    if not d.invoice_number:
        issues.append("Missing invoice number")
    if not d.due_date and d.document_type == "invoice":
        issues.append("Missing due date")

    status = "✓ VALID" if not issues else f"⚠ {len(issues)} issue(s)"
    print(f"\n{inv['file']}: {status}")
    for issue in issues:
        print(f"  → {issue}")

## Step 5 — Export & Summary

In [ ]:
# Export to structured format
Path("sample_data").mkdir(exist_ok=True)
export = [{"file": e["file"], **e["data"].model_dump()} for e in extracted if e["data"]]
with open("sample_data/extracted_invoices.json", "w") as f:
    json.dump(export, f, indent=2, default=str)

# Summary table
rows = []
for e in extracted:
    if e["data"]:
        rows.append({
            "file": e["file"], "vendor": e["data"].vendor_name,
            "type": e["data"].document_type, "total": e["data"].total,
            "items": len(e["data"].line_items),
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nTotal across all documents: ${df['total'].sum():,.2f}")

## What You Learned
- **OCR → structured extraction** with Pydantic schemas
- **Line-item parsing** from messy text
- **Math validation** (subtotal, tax, discount, total)
- **Anomaly detection** and data quality checks